# Prueba de cobertura del validador

Notebook armado para incumplir a proposito las reglas obligatorias.
No debe ejecutarse: existe solo para verificar la deteccion.

In [ ]:
# Carga de desembolsos
# Cabecera intencionalmente incompleta
# Solo trae dos etiquetas de las seis exigidas

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType, StructField, StructType
from pyspark.sql.functions import udf, col

import logging
import time

logger = logging.getLogger("PRUEBA")
logger.setLevel(logging.INFO)

In [ ]:
var_catalogo = "mb_silver_prod"
var_fecha = "2026-08-30"

ruta_bronce = "abfss://bronze@almacenamiento.blob.core.windows.net/entrada/"
ruta_plata = "abfss://silver@otrocontenedor.blob.core.windows.net/salida/"
ruta_oro = "abfss://gold@cuentaequivocada.blob.core.windows.net/publicado/"
ruta_puente = "/intercambio/MB_PERU/ENTRADA/sistema"

print("Rutas configuradas")

In [ ]:
esquema_desembolso = StructType([
    StructField("cod_operacion", StringType()),
    StructField("fec_desembolso", StringType()),
])

@udf(StringType())
def limpiar_moneda(valor):
    return valor.strip().upper()

In [ ]:
def leer_desembolsos(fecha):
    return spark.sql("select * from mb_silver_prod.ope.h_desembolso")


df_desembolso = leer_desembolsos(var_fecha)
filas = df_desembolso.collect()

logger.info("Se leyeron los desembolsos")

In [ ]:
(df_desembolso
    .write
    .format("csv")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .save(ruta_oro))

logger.info("Escritura terminada")

In [ ]:
%sql
CREATE TABLE mb_silver_prod.ope.desembolso_consolidado (
    cod_operacion STRING,
    fec_desembolso DATE
) USING DELTA;

In [ ]:
%sql
CREATE TABLE dwh_prod.operaciones.TablaDeDesembolsosConsolidadaHistoricaDelAreaDeOperacionesFinancierasÑ (
    CodOperacion STRING,
    documento_de_identidad_del_cliente_titular_principal STRING,
    identificación STRING,
    saldo_pendiente STRING,
    mto_desembolso DOUBLE,
    nom_cliente_dac STRING
);